# 원하는 포즈로 이미지 만드는 도구 (Pose Image Tool)

참조 사진 한 장에서 **OpenPose**로 사람 관절(스켈레톤)을 뽑아내고, 그 정보를 **ControlNet** 조건으로 넣어서 "같은 자세, 다른 인물/장면" 이미지를 생성하는 도구입니다.

**흐름**: 참조 사진 → OpenPose 관절 추출 → ControlNet 조건 + 프롬프트 → 이미지 생성 → 저장

이 노트는 처음 보는 사람도 셀을 위에서 아래로 순서대로 실행하면 그대로 동작하도록 만들었습니다. 각 셀 맨 위 주석에 그 셀이 하는 일을 적어 두었습니다.

## 왜 FLUX.1-dev + ControlNet-Union을 쓰는가

과제 원안은 FLUX.2-klein 4B를 기준으로 하지만, 이 노트를 작성한 시점(2026년 9월) 기준으로 **FLUX.2-klein용 공식 ControlNet 포즈 체크포인트는 diffusers에 아직 안정적으로 올라와 있지 않습니다** (커뮤니티용 LoRA/ComfyUI 워크플로만 일부 존재). 반면 **FLUX.1-dev**는 diffusers 공식 문서가 `FluxControlNetPipeline` + `FluxControlNetModel`을 통한 ControlNet 사용법을 명시하고 있고, `InstantX/FLUX.1-dev-Controlnet-Union` 체크포인트가 포즈(control_mode=4)를 포함한 여러 조건을 지원합니다.

과제 안내에 "FLUX.2-klein 4B 또는 비슷한 모델"이라는 여지가 있어, 이 노트는 **FLUX.1-dev + InstantX Union ControlNet** 조합을 사용합니다. FLUX.1-dev는 같은 Black Forest Labs의 FLUX 계열 모델이며, 사용을 위해 Hugging Face에서 라이선스 동의와 토큰 발급이 한 번 필요합니다 (아래 3번 셀 참고).

참고 자료:
- https://huggingface.co/docs/diffusers/en/api/pipelines/controlnet_flux
- https://huggingface.co/InstantX/FLUX.1-dev-Controlnet-Union
- https://huggingface.co/black-forest-labs/FLUX.2-klein-4B (참고용, 이번 노트에서는 미사용)

In [1]:
# 이 셀이 하는 일: Colab 런타임에 필요한 라이브러리를 설치합니다.
# diffusers는 FLUX ControlNet 지원이 비교적 최근에 추가되었으므로 GitHub 최신 버전을 설치합니다.
!pip install -q git+https://github.com/huggingface/diffusers.git
!pip install -q transformers accelerate safetensors sentencepiece controlnet_aux mediapipe huggingface_hub Pillow

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
!pip install -q -U torchao

In [7]:
import mediapipe as mp
import types

if not hasattr(mp, "solutions"):
    mp.solutions = types.SimpleNamespace(
        drawing_utils=types.SimpleNamespace(),
        drawing_styles=types.SimpleNamespace(),
        face_detection=types.SimpleNamespace(),
    )

In [ ]:
# 이 셀이 하는 일: 필요한 모듈을 임포트하고, GPU가 잡혔는지 확인합니다.
import os
import torch
from PIL import Image

from diffusers import FluxControlNetModel, FluxControlNetPipeline
from diffusers.utils import load_image
from controlnet_aux import OpenposeDetector

print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("경고: GPU가 없습니다. Colab 메뉴에서 런타임 > 런타임 유형 변경 > GPU(T4 이상)로 바꾸세요.")

# 결과물을 저장할 폴더 (GitHub 제출용 samples/ 폴더와 이름을 맞춥니다)
SAMPLES_DIR = "samples"
os.makedirs(SAMPLES_DIR, exist_ok=True)

In [ ]:
# 이 셀이 하는 일: Hugging Face에 로그인합니다.
# FLUX.1-dev는 gated 모델이라 (1) https://huggingface.co/black-forest-labs/FLUX.1-dev 에서 라이선스 동의,
# (2) https://huggingface.co/settings/tokens 에서 토큰 발급이 먼저 필요합니다.
from huggingface_hub import login

HF_TOKEN = ""  # 여기에 본인 Hugging Face 토큰을 붙여넣거나, 비워두면 아래에서 입력창이 뜹니다.

if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    login()  # 토큰 입력창이 뜹니다.

## 1단계 — 포즈 조건을 넣어 도구 만들기

In [ ]:
# 이 셀이 하는 일: 참조 사진을 불러오는 헬퍼 함수를 정의합니다.
# URL이든 로컬 파일 경로든 그대로 받아서 PIL 이미지로 반환하고,
# FLUX가 요구하는 대로 가로/세로를 16의 배수로 맞춰 리사이즈합니다.

def load_reference_image(source: str, max_side: int = 1024) -> Image.Image:
    image = load_image(source).convert("RGB")

    w, h = image.size
    scale = min(max_side / max(w, h), 1.0)
    w, h = int(w * scale), int(h * scale)

    # FLUX 파이프라인은 8~16의 배수 크기를 기대하므로 반올림합니다.
    w = max(16, (w // 16) * 16)
    h = max(16, (h // 16) * 16)
    return image.resize((w, h))


# 기본 데모용 참조 사진 (diffusers 공식 ControlNet 튜토리얼에서 쓰는 예시 사진).
# 본인 사진으로 테스트하려면 이 URL 대신 Colab에 업로드한 파일 경로(예: "pose_01.jpg")를 넣으면 됩니다.
POSE_01_SOURCE = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/person.png"

reference_image_01 = load_reference_image(POSE_01_SOURCE)
reference_image_01

In [ ]:
# 이 셀이 하는 일: OpenPose 검출기를 불러오고, 참조 사진에서 관절 스켈레톤을 추출합니다.
# 추출된 스켈레톤 이미지가 ControlNet에 들어갈 '포즈 조건'입니다.

openpose_detector = OpenposeDetector.from_pretrained("lllyasviel/ControlNet")


def extract_pose(reference_image: Image.Image) -> Image.Image:
    return openpose_detector(reference_image, hand_and_face=False)


pose_skeleton_01 = extract_pose(reference_image_01)
pose_skeleton_01.save(os.path.join(SAMPLES_DIR, "pose_01.png"))
pose_skeleton_01

In [ ]:
# 이 셀이 하는 일: ControlNet(포즈)과 FLUX.1-dev 파이프라인을 로드합니다.
# T4 같은 VRAM이 작은 GPU에서도 돌아가도록 model_cpu_offload를 켭니다.

BASE_MODEL = "black-forest-labs/FLUX.1-dev"
CONTROLNET_MODEL = "InstantX/FLUX.1-dev-Controlnet-Union"
POSE_CONTROL_MODE = 4  # Union ControlNet의 control_mode: 0=canny 1=tile 2=depth 3=blur 4=pose 5=gray 6=lq

controlnet = FluxControlNetModel.from_pretrained(CONTROLNET_MODEL, torch_dtype=torch.bfloat16)
pipe = FluxControlNetPipeline.from_pretrained(
    BASE_MODEL,
    controlnet=controlnet,
    torch_dtype=torch.bfloat16,
)
pipe.enable_model_cpu_offload()  # GPU 메모리가 넉넉하면 pipe.to("cuda")로 바꿔도 됩니다.

In [ ]:
# 이 셀이 하는 일: '포즈 + 프롬프트 -> 이미지' 전체 과정을 하나의 함수(도구)로 묶습니다.
# 이 함수 하나가 이번 과제의 핵심 도구입니다: 참조 사진과 원하는 프롬프트만 주면 같은 자세의 새 이미지를 만들어 줍니다.

def generate_pose_image(
    pose_reference_source: str,
    prompt: str,
    output_name: str,
    negative_prompt: str = "blurry, low quality, distorted anatomy, extra limbs",
    controlnet_conditioning_scale: float = 0.7,
    num_inference_steps: int = 28,
    guidance_scale: float = 3.5,
    seed: int = 0,
):
    reference_image = load_reference_image(pose_reference_source)
    pose_image = extract_pose(reference_image)
    pose_image.save(os.path.join(SAMPLES_DIR, f"{output_name}_pose.png"))

    generator = torch.Generator(device="cpu").manual_seed(seed)
    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        control_image=pose_image,
        control_mode=POSE_CONTROL_MODE,
        width=pose_image.width,
        height=pose_image.height,
        controlnet_conditioning_scale=controlnet_conditioning_scale,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=generator,
    ).images[0]

    output_path = os.path.join(SAMPLES_DIR, f"{output_name}.png")
    result.save(output_path)
    print("저장 완료:", output_path)
    return result, pose_image

In [ ]:
# 이 셀이 하는 일: 도구를 처음으로 한 번 돌려서 기준(baseline) 결과를 만듭니다.
# 결과는 samples/output_01.png 로 저장됩니다.

baseline_prompt = "a photo of an astronaut standing on the moon, cinematic lighting, highly detailed"

output_image_01, _ = generate_pose_image(
    pose_reference_source=POSE_01_SOURCE,
    prompt=baseline_prompt,
    output_name="output_01",
    seed=0,
)
output_image_01

## 2단계 — 다른 포즈와 프롬프트로 바꿔보기

(1) 같은 포즈, 다른 프롬프트 3개 / (2) 같은 프롬프트, 다른 포즈 사진 하나. 어느 쪽을 바꿀 때 결과가 크게 달라지는지 관찰합니다.

실제 프롬프트 목록은 [prompts.md](prompts.md)에도 정리되어 있습니다.

In [ ]:
# 이 셀이 하는 일: 실험 A - 포즈는 pose_01로 고정하고, 프롬프트만 바꿔가며 3장을 생성합니다.

prompt_variants = [
    "a knight in shining armor standing in a medieval castle courtyard, dramatic lighting",
    "a ballet dancer in a white tutu on a theater stage, spotlight, soft focus background",
    "a robot made of chrome metal standing in a futuristic city street at night, neon lights",
]

experiment_a_results = []
for i, prompt in enumerate(prompt_variants, start=1):
    image, _ = generate_pose_image(
        pose_reference_source=POSE_01_SOURCE,
        prompt=prompt,
        output_name=f"expA_samepose_prompt{i}",
        seed=i,
    )
    experiment_a_results.append(image)

In [ ]:
# 이 셀이 하는 일: 실험 B - 프롬프트는 고정하고, pose_01과는 다른 자세의 참조 사진(pose_02)을 넣어 결과를 비교합니다.
# pose_02는 과제 취지에 맞게 '직접 고른 사람 사진'을 씁니다. Colab에서 실행 중이면 업로드 창이 뜹니다.
# 로컬 Jupyter 등 Colab이 아닌 환경이면 POSE_02_SOURCE에 이미지 경로나 URL을 직접 입력하세요.

try:
    from google.colab import files
    print("pose_01과는 다른 자세가 뚜렷한 사람 사진을 업로드하세요 (예: 팔을 벌리거나 앉아 있는 사진).")
    uploaded = files.upload()
    POSE_02_SOURCE = next(iter(uploaded))
except ModuleNotFoundError:
    POSE_02_SOURCE = "pose_02_input.jpg"  # Colab이 아니면 이 경로에 본인 사진을 넣어두세요.

fixed_prompt = "a superhero in a dynamic pose, comic book art style, bold colors"

reference_image_02 = load_reference_image(POSE_02_SOURCE)
pose_skeleton_02 = extract_pose(reference_image_02)
pose_skeleton_02.save(os.path.join(SAMPLES_DIR, "pose_02.png"))

output_image_02, _ = generate_pose_image(
    pose_reference_source=POSE_02_SOURCE,
    prompt=fixed_prompt,
    output_name="output_02",
    seed=0,
)

# 같은 프롬프트를 pose_01에도 적용해서 나란히 비교합니다.
output_image_01_same_prompt, _ = generate_pose_image(
    pose_reference_source=POSE_01_SOURCE,
    prompt=fixed_prompt,
    output_name="expB_samepromt_pose01",
    seed=0,
)
output_image_02

In [ ]:
# 이 셀이 하는 일: 지금까지 만든 결과들을 한 번에 그리드로 띄워서 비교하기 쉽게 보여줍니다.
import matplotlib.pyplot as plt

rows = [
    ("기준 (pose_01)", [reference_image_01, pose_skeleton_01, output_image_01]),
    ("실험 A (같은 포즈, 다른 프롬프트)", experiment_a_results),
    ("실험 B (pose_01 vs pose_02, 같은 프롬프트)", [output_image_01_same_prompt, output_image_02]),
]

for title, images in rows:
    fig, axes = plt.subplots(1, len(images), figsize=(4 * len(images), 4))
    if len(images) == 1:
        axes = [axes]
    fig.suptitle(title)
    for ax, img in zip(axes, images):
        ax.imshow(img)
        ax.axis("off")
    plt.show()

## 관찰 결과 (2단계 정리)

> 아래는 실제로 위 셀들을 Colab에서 실행한 뒤 관찰한 내용으로 채워 넣는 칸입니다. (Colab에서 실행 후 작성)

- **같은 포즈, 다른 프롬프트 (실험 A)**: (실행 후 작성 — 예: 자세/구도는 유지되지만 인물·배경·스타일은 프롬프트를 크게 따라갔는가?)
- **같은 프롬프트, 다른 포즈 (실험 B)**: (실행 후 작성 — 예: 팔다리 각도, 몸의 방향이 포즈 사진을 얼마나 그대로 따라갔는가?)
- **어느 쪽을 바꿨을 때 결과가 크게 달라졌는가**: (실행 후 작성)
- **잘 안 따라온 부분이 있다면**: (실행 후 작성 — 예: 손가락/얼굴 디테일, 조명 방향, controlnet_conditioning_scale 값에 따른 차이 등)